In [ ]:
!pip -q install transformers datasets accelerate

import torch, torch.nn as nn, numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# data: stars -> 3 classes
ds = load_dataset("SetFit/amazon_reviews_multi_en")
lab = lambda x: {"label": 0 if x["label"] <= 1 else (1 if x["label"] == 2 else 2)}
tr = ds["train"].shuffle(seed=42).select(range(8000)).map(lab)
te = ds["test"].shuffle(seed=42).select(range(2000)).map(lab)

name = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(name)
enc = lambda d: d.map(lambda b: tok(b["text"], truncation=True, padding="max_length", max_length=128), batched=True)
tr_e, te_e = enc(tr), enc(te)

# BERT (DistilBERT)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3)
t = Trainer(model=model, train_dataset=tr_e,
            args=TrainingArguments("out", num_train_epochs=2, save_strategy="no", report_to="none"))
t.train()
bert_acc = (t.predict(te_e).predictions.argmax(-1) == np.array(te["label"])).mean()

# LSTM baseline (PyTorch)
class LSTM(nn.Module):
    def __init__(s):
        super().__init__()
        s.emb = nn.Embedding(tok.vocab_size, 64)
        s.lstm = nn.LSTM(64, 64, batch_first=True)
        s.fc = nn.Linear(64, 3)
    def forward(s, x):
        return s.fc(s.lstm(s.emb(x))[1][0][-1])

dev = "cuda" if torch.cuda.is_available() else "cpu"
m = LSTM().to(dev)
opt = torch.optim.Adam(m.parameters(), 1e-3)
X, y = torch.tensor(tr_e["input_ids"]), torch.tensor(tr_e["label"])
for ep in range(3):
    for i in range(0, len(X), 64):
        opt.zero_grad()
        loss = nn.functional.cross_entropy(m(X[i:i+64].to(dev)), y[i:i+64].to(dev))
        loss.backward(); opt.step()

with torch.no_grad():
    p = m(torch.tensor(te_e["input_ids"]).to(dev)).argmax(-1).cpu().numpy()
lstm_acc = (p == np.array(te["label"])).mean()

print("BERT accuracy:", bert_acc, "| LSTM accuracy:", lstm_acc)


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

C:\Users\IT-phulpur\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\IT-phulpur\.cache\huggingface\hub\datasets--SetFit--amazon_reviews_multi_en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\IT-phulpur\AppData\Local\Programs\Python\Python310\lib\site-pa

train.jsonl: reconstructing file:   0%|          |  0.00B / 47.4MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\IT-phulpur\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory 

Step,Training Loss
500,0.759200
1000,0.660904
1500,0.479332
